# Chronos-2 MASE Scaffold

Scaffolds Chronos-2 into the MASE framework.

**What this notebook does:**
1. Loads the pretrained Chronos-2 model
2. Builds dummy inputs matching the model's expected signature
3. Traces it into a `MaseGraph`
4. Runs a forward pass through the graph to verify correctness

**Useful as:** a starting point for adding MASE transformation/optimisation passes.

## 1. Imports & config

In [1]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ.setdefault('HOME', os.environ.get('USERPROFILE', str(Path.home())))

import torch

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 2

print(f'Device: {DEVICE}')
print(f'Batch size: {BATCH_SIZE}')

Device: cuda
Batch size: 2


## 2. Load Chronos-2 model

In [2]:
from chop.models import get_model

model = get_model('chronos-2', pretrained=True)
model.eval()
model = model.to(DEVICE)

chronos_cfg = model.config.chronos_config
C_LEN       = chronos_cfg['context_length']    # e.g. 8192
OUT_PATCH   = chronos_cfg['output_patch_size'] # e.g. 16
N_VARIATES = 20

print('Model loaded:', type(model).__name__)
print('Context length:', C_LEN)
print('Output patch size:', OUT_PATCH)
print('\nChronos config:')
for k, v in chronos_cfg.items():
    print(f'  {k}: {v}')

Model loaded: Chronos2Model
Context length: 8192
Output patch size: 16

Chronos config:
  context_length: 8192
  input_patch_size: 16
  input_patch_stride: 16
  max_output_patches: 64
  output_patch_size: 16
  quantiles: [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.99]
  time_encoding_scale: 8192
  use_arcsinh: True
  use_reg_token: True


## 3. Build dummy inputs

- `context` — the historical time series values `(batch, context_length)` where batch = n_variates
- `group_ids` — which series in the batch share information via group attention.
  `zeros` puts everything in group 0 (fully multivariate), exercising the full group attention path.
  `arange` would make every series independent (diagonal mask, no cross-series mixing).
- `future_covariates` — known future values; `NaN` means no known future (pure forecast target)
- `num_output_patches` — how many output patches to predict

In [3]:
dummy_in = {
    'context':            torch.randn((N_VARIATES, C_LEN), device=DEVICE),
    'group_ids':          torch.zeros((N_VARIATES,), dtype=torch.long, device=DEVICE),
    # 'future_covariates':  torch.zeros((N_VARIATES, OUT_PATCH), device=DEVICE),
    'num_output_patches': OUT_PATCH,
}

In [4]:
print(model.encoder.block[0].layer[1])

GroupSelfAttention(
  (self_attention): MHA(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
  )
  (layer_norm): Chronos2LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
)


In [7]:
from chop import MaseGraph
from chop.models.chronos2.modeling_chronos2 import GroupSelfAttention, TimeSelfAttention

mg = MaseGraph(
    model=model, 
    hf_input_names=list(dummy_in.keys()),
    custom_ops={
        "modules":{
            GroupSelfAttention : {},
            TimeSelfAttention: {},
        },
        "functions":{}
    })
# mg = MaseGraph(
#     model=model, 
#     hf_input_names=list(dummy_in.keys()),)


target = []
for node in mg.nodes:
    if node.op == "call_module":
        mod = mg.model.get_submodule(node.target)
        if "GroupSelfAttention" in type(mod).__name__:
            target.append(mod)


print(target)
        



[GroupSelfAttention(
  (self_attention): MHA(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
  )
  (layer_norm): Chronos2LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
), GroupSelfAttention(
  (self_attention): MHA(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)
    (o): Linear(in_features=768, out_features=768, bias=False)
  )
  (layer_norm): Chronos2LayerNorm()
  (dropout): Dropout(p=0.1, inplace=False)
), GroupSelfAttention(
  (self_attention): MHA(
    (q): Linear(in_features=768, out_features=768, bias=False)
    (k): Linear(in_features=768, out_features=768, bias=False)
    (v): Linear(in_features=768, out_features=768, bias=False)

## 3b. Dummy input cases

Different `group_ids` / `future_covariates` combinations exercise different model paths:

| Case | `group_ids` | `future_covariates` | What it represents |
|---|---|---|---|
| `univariate` | `arange` (all unique) | NaN | Every series forecast independently — no cross-series attention |
| `multivariate` | all zeros | NaN | All series in one group — full cross-series attention |
| `mixed` | half-half | NaN | Two groups of equal size — partial cross-series mixing |
| `with_covariates` | all zeros | alternating NaN / real | Half the batch are known covariates, half are targets |

## 8. Speed Benchmark

Compares the original model against the traced `mg.model` (fx.GraphModule), and provides a template slot for a transformed model.

Uses CUDA events for accurate GPU timing. Run this cell before and after applying a transformation pass to measure speedup.

In [ ]:
import statistics

# ── Benchmark helper ──────────────────────────────────────────────────────────
def benchmark_model(fn, inputs: dict, warmup: int = 5, iters: int = 20) -> float:
    """Returns median latency in milliseconds. Uses CUDA events on GPU."""
    use_cuda = torch.cuda.is_available()
    with torch.no_grad():
        for _ in range(warmup):
            fn(**inputs)
        if use_cuda:
            torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(iters):
            if use_cuda:
                start = torch.cuda.Event(enable_timing=True)
                end   = torch.cuda.Event(enable_timing=True)
                start.record()
                fn(**inputs)
                end.record()
                torch.cuda.synchronize()
                times.append(start.elapsed_time(end))
            else:
                import time
                t0 = time.perf_counter()
                fn(**inputs)
                times.append((time.perf_counter() - t0) * 1e3)

    return statistics.median(times)


mg.model.chronos_config = model.chronos_config

# ── Run across all dummy cases ────────────────────────────────────────────────
print(f"{'case':<20} {'baseline (ms)':>14} {'mg.model (ms)':>14} {'speedup':>9}")
print("-" * 62)

results = {}
for case_name, inputs in dummy_cases.items():
    base_ms  = benchmark_model(model,    inputs)
    graph_ms = benchmark_model(mg.model, inputs)
    results[case_name] = {'baseline': base_ms, 'graph': graph_ms}
    print(f"{case_name:<20} {base_ms:>14.2f} {graph_ms:>14.2f} {base_ms/graph_ms:>8.3f}x")

# ── Template: benchmark your transform pass ───────────────────────────────────
# After applying your pass:
#
#   mg, _ = my_transform_pass(mg, pass_args={...})
#   mg.model.recompile()
#   mg.fx_graph.lint()
#
#   print(f"\n{'case':<20} {'baseline':>12} {'transformed':>12} {'speedup':>9}")
#   for case_name, inputs in dummy_cases.items():
#       t_ms = benchmark_model(mg.model, inputs)
#       base_ms = results[case_name]['baseline']
#       print(f"{case_name:<20} {base_ms:>11.2f}ms {t_ms:>11.2f}ms {base_ms/t_ms:>8.3f}x")
